# SPOD to Mapping Excel
- Prerequisites: 
  - Anaconda packages: `pandas, openpyxl, seaborn`


## Result

Excel sheet containing:

Sheet with all Datapoints (Systems, Tables and Columns) mapped against the Information Model (Entity, Attribute)
Table covering the overview sheet


Optional:
Sheet per System - IM containing sample data 

## Structure

1. Define mapping between SPOD (json) and columns in the resulting Excel sheet


## Configuration

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

#MODEL_SOURCE = '/Users/bue/projects/sika/Sika-IM/DB/Sika-IM.json'
MODEL_SOURCE = '/Users/bue/projects/geberit/DEAP/DB/IM_GEBERIT.json'

DESTINATION = 'geberit.xlsx'

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
import matplotlib.colors as mcolors
import seaborn as sns

In [ ]:
# openpyxl
from openpyxl import Workbook
from openpyxl.worksheet.table import Table
from openpyxl.utils.cell import get_column_letter
from openpyxl.styles import PatternFill

In [ ]:
spod_file = Path(MODEL_SOURCE)
assert spod_file.is_file(), f"Cannot find SPOD file '{configfile.resolve()}'"

with open(spod_file, 'r') as src:
    spod = json.load(src)
assert spod['model'] is not None
print(f"Loaded SPOD containing {spod['model']} from '{spod_file.resolve()}'")

In [ ]:
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
print(f"Languages: {list(spod['languages'].keys())}")
mapdict = {}
for entry in ['entities', 'attributes', 'systems', 'columns']:
    print(f"- {entry}: {len(spod[entry])}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.excel.mapping_publisher import generate
from SSOT_infra.translator import Translator

## Translation shortcut tr

In [ ]:
translator = Translator('de')

## List structure definition

In [ ]:
columns_mapped = {}
system_index = {}

In [ ]:
headings_im = ["FQN", "EID", "AID", "Entity (EN) ", "Attribute (EN)", "Entity (DE) ", "Attribute (DE)", "Entity (FR) ", "Attribute (FR)", "Examples", "Description", "#"]

In [ ]:
len(headings_im)

In [ ]:
def emit_system_columns(spod: dict) -> [str]:
    result = []
    for key, system in spod['systems'].items():
        system_index[key] = len(headings_im) + len(result)
        result.append(key + ':FQN')
        result.append(system['name'])
        result.append(key + ':REF')
    return result

In [ ]:
systems = emit_system_columns(spod)

In [ ]:
systems

In [ ]:
def export_column(key: str, column: dict) -> []:
    result = [
                column['interface-id+'] + ':' + column['table-id'] + ':' + key,
                column['name'],
                column['interface_col_id'],
            ]
    return result
    
def firsthit(spod: dict, attribute_key: str, system_key: str) -> []:
    for ckey, column in spod['columns'].items():
        if system_key == column['interface-id+'] and attribute_key in column['attributesmapped']:
            result = export_column(ckey, column)
            columns_mapped[ckey] = attribute_key
            return result
    return [ '', '', '' ]

In [ ]:
def emit_row(spod: dict, attribute_key: str, attribute: dict, translator: Translator) -> []:
    enti_key = attribute['entity']
    entity = spod['entities'].get(enti_key)
    assert entity is not None, f"Missing entity {enti_key}"
    result = [
        enti_key + ':' + attribute_key,
        enti_key,
        attribute_key,
        translator.tr(entity['name'], 'en'),
        translator.tr(attribute['name'], 'en'),
        translator.tr(entity['name'], 'de'),
        translator.tr(attribute['name'], 'de'),
        translator.tr(entity['name'], 'fr'),
        translator.tr(attribute['name'], 'fr'),
        ', '.join(translator.tr(attribute.get('examples'), 'en')),
        translator.tr(attribute['descr'], 'en'),
        len(attribute['columnsmapped+']),
    ]

    for skey in spod['systems'].keys():
        mapping = firsthit(spod, attribute_key, skey)
        result = result + mapping

    return result

In [ ]:
headings = headings_im + systems
f"Columns ({len(headings)}): {', '.join(headings)}"

# Create data table (content)

In [ ]:
data_table = [ emit_row(spod, key, attribute, translator) for key, attribute in spod['attributes'].items() ]

In [ ]:
len(data_table)

In [ ]:
f"Already mapped {len(columns_mapped)} columns of {len(spod['columns'])}"

## Append unmapped columns to the bottom

In [ ]:
def aux_row(key: str, column: dict) -> []:
    result = [ None ] * len(headings)
    if len(column['attributesmapped']) > 0:
        result[len(headings_im) - 1] = 1
    else:
        result[len(headings_im) - 1] = 0
    index = system_index[column['interface-id+']]
    values = export_column(key, column)
    result[index + 0] = values[0]
    result[index + 1] = values[1]
    result[index + 2] = values[2]
    return result

In [ ]:
remainder = [ aux_row(key, spod['columns'][key]) for key in filter(lambda key: key not in columns_mapped.keys(), spod['columns'].keys()) ]

In [ ]:
data_table = data_table + remainder

# Prepare Excel Workbook

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = 'Mapping'

# add column headings. NB. these must be strings
ws.append(headings)
for row in data_table:
    ws.append(row)

## Define a data table readable by Sharepoint

In [ ]:
tab = Table(displayName="Mapping", ref=f"A1:{get_column_letter(len(headings))}{len(data_table)+1}")
ws.add_table(tab)
tab._initialise_columns()

for column, value in zip(tab.tableColumns, headings):
    column.name = value

## Styling

In [ ]:
pal = list(sns.color_palette('pastel'))

for column_index in range(len(headings_im), len(headings)):
    color_index = int((column_index - len(headings_im)) / 3)
    color = pal[color_index % len(pal)]
    rgb = str(mcolors.to_hex(color))[1:]
    #print(rgb)
    for cell in ws[get_column_letter(column_index + 1)]:
        cell.fill = PatternFill(fgColor=rgb, fill_type = "solid")

### Resize and hide columns

In [ ]:
ws.column_dimensions['B'].hidden= True
ws.column_dimensions['C'].hidden= True

ws.column_dimensions['F'].hidden= True
ws.column_dimensions['G'].hidden= True
ws.column_dimensions['H'].hidden= True
ws.column_dimensions['I'].hidden= True


In [ ]:
base = len(headings_im)
index = 0
for system in spod['systems'].values():
    colnr = base + 1 + (index * 3)
    col_letter = get_column_letter(colnr)
    dim = ws.column_dimensions[col_letter]
    dim.hidden= True
    
    col_letter_cid = get_column_letter(colnr + 1)
    dim = ws.column_dimensions[col_letter_cid]
    dim.hidden= True
    
    print(f"Hiding columns {col_letter} ({ws[col_letter + '1'].value})"
          f" and {col_letter_cid} ({ws[col_letter_cid + '1'].value}) on System {system['name']} idx {colnr}")
    
    index += 1

## Save to Excel file

In [ ]:
wb.save(DESTINATION)
print(f"Wrote {DESTINATION}")

# Visually verify

In [ ]:
import pandas

excel_data_df = pandas.read_excel(DESTINATION, sheet_name='Mapping')

In [ ]:
from IPython.display import display, HTML
display(excel_data_df)